# 05 · Let the graph run the workflow

A LangGraph loop lets the model choose the next tool and stop when it can
answer. The order lookup is local; policy search is an actual MCP tool backed
by Knowledge Bases or Chroma.

**Suggested delivery:** a prepared closing walkthrough, with the complete code provided for follow-up.

**Kernel:** select **Python (Bedrock inference)**, or this repository's `.venv/bin/python` in VS Code. The local MCP server uses the same Python interpreter as this notebook.

In [ ]:
import json
from pathlib import Path
from typing import TypedDict
from IPython.display import display
from langgraph.graph import END, START, StateGraph
from utils.workshop import Rates, Run, Settings, compare
from utils.rag_mcp import PolicyMCPClient

settings = Settings.from_env()
rates = Rates.load(model_id=settings.model_id)
order = json.loads(Path("data/order.json").read_text())
MAX_MODEL_CALLS = 5
USE_PROMPT_CACHE = True
cache_reference = None
if USE_PROMPT_CACHE:
    cache_reference = Path("data/cache_reference.md").read_text()

## Edit the prompts

`system_prompt` defines the agent's behavior and `instruction` supplies its starting task. Both enter fresh graph state at invocation, so editing them does not require recompiling the graph.

Rerun this cell after edits, then rerun the relevant example. For comparisons, rerun both baseline and tuned versions.

In [ ]:
system_prompt = (
    "Investigate the order using get_order_status, then use "
    "search_support_policy for current policy evidence. "
    "Answer in at most three sentences with source labels. "
    "Do not claim to create a replacement or refund."
)
instruction = "Check ORDER-1042 and tell me if a replacement is allowed."

## Define the local tool

As in notebook 01, we expose the input contract and return only useful fields.
The policy tool's schema will be discovered from the MCP server.

In [ ]:
order_tool = {
    "name": "get_order_status",
    "description": "Look up an order's current status and next support step.",
    "parameters": {
        "type": "object",
        "properties": {"order_id": {"type": "string"}},
        "required": ["order_id"],
        "additionalProperties": False,
    },
}

def lookup_order(order_id):
    if order_id != order["order_id"]:
        raise ValueError("Use the known order ID from the demo.")
    fields = ["order_id", "status", "days_late", "next_step"]
    return {field: order[field] for field in fields}

## Model node: decide what is needed next

The graph retains messages and the latest tool requests. It uses the same
inference wrapper as the earlier notebooks, including measurements and retries.

In [ ]:
class AgentState(TypedDict):
    system_prompt: str
    messages: list
    tool_calls: list
    turns: int
    answer: str

def agent_node(state):
    if state["turns"] >= MAX_MODEL_CALLS:
        raise RuntimeError("Agent model-call budget exhausted.")
    response = agent_run.generate(
        f"agent_{state['turns'] + 1}", state["messages"],
        tools=agent_tools, max_output_tokens=2048,
        cache_prefix=cache_reference,
        instructions=state["system_prompt"],
    )
    return {
        "messages": [*state["messages"], response.message],
        "tool_calls": list(response.tool_calls),
        "turns": state["turns"] + 1,
        "answer": response.text,
    }

## Tool node: execute the model's requests

Policy calls cross the MCP stdio connection; the model never accesses the
database directly. Results are appended using the same portable message
format as notebook 01.

In [ ]:
def tool_node(state):
    messages = list(state["messages"])
    for call in state["tool_calls"]:
        if call.name == "get_order_status":
            result = agent_run.call(
                call.name, "tool",
                lambda timeout: lookup_order(**call.arguments),
            )
        elif call.name == "search_support_policy":
            result = agent_run.call(
                call.name, "retrieval",
                lambda timeout: policy_client.call_tool(
                    call.name, call.arguments, timeout
                ),
            )
        else:
            raise ValueError(f"Unknown tool: {call.name}")
        messages.append({
            "role": "tool", "tool_call_id": call.id,
            "content": json.dumps(result),
        })
    return {"messages": messages, "tool_calls": []}

def next_step(state):
    return "tools" if state["tool_calls"] else "done"

## Wire the loop

`START → agent → tools → agent` repeats while the model requests tools.
A text answer routes to `END`; both graph recursion and model-call budgets are bounded.

In [ ]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent", next_step, {"tools": "tools", "done": END}
)
builder.add_edge("tools", "agent")
agent_graph = builder.compile()

## Connect MCP, then run one task

MCP startup and local indexing happen before the timed run. With no Knowledge
Base ID, the server uses its local Chroma index and embeddings; model calls
still go to Bedrock.

In [ ]:
with PolicyMCPClient(settings) as policy_client:
    agent_tools = [order_tool, *policy_client.tools()]
    print("Available tools:", [tool["name"] for tool in agent_tools])
    initial_state = {
        "system_prompt": system_prompt,
        "messages": [{
            "role": "user",
            "content": instruction,
        }],
        "tool_calls": [], "turns": 0, "answer": "",
    }
    with Run("05_langgraph", settings, rates) as agent_run:
        result = agent_graph.invoke(
            initial_state,
            config={"recursion_limit": 2 * MAX_MODEL_CALLS + 1},
        )
print(result["answer"])
display(agent_run.metrics())
display(compare(agent_run))
agent_run.save()

**Review the trace:** did the model obtain order facts and policy evidence
before answering, and does its citation support the decision? The graph
automates control flow; it does not make the answer correct automatically.

The agent keeps compact tool results and a bounded answer, and combines
planning with tool selection. Enable `USE_PROMPT_CACHE` to reuse the long
workflow reference from notebook 04 when that context is useful.